# 01. Data Understanding

## Objective

The goal of this notebook is to understand the MovieLens dataset
before performing data preprocessing or building recommendation models.

We will examine:

- Dataset structure
- Number of users and movies
- Rating distribution
- Movie metadata
- User-generated tags
- Relationships between datasets
- Missing values
- Duplicate records
- Timestamp coverage
- Basic data quality issues

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define the project and raw data paths

PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)

Project directory: c:\Users\moham\Desktop\Netflix Movie recomendation system
Raw data directory: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\raw


In [3]:
# List all files and folders inside the raw data directory

for item in RAW_DATA_DIR.iterdir():
    print(item.name)

ml-32m


In [4]:
# Define the MovieLens 32M dataset directory

MOVIELENS_DIR = RAW_DATA_DIR / "ml-32m"

print("MovieLens directory:", MOVIELENS_DIR)
print("Exists:", MOVIELENS_DIR.exists())

MovieLens directory: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\raw\ml-32m
Exists: True


In [5]:
# List the files inside the MovieLens dataset

for item in MOVIELENS_DIR.iterdir():
    print(item.name)

checksums.txt
links.csv
movies.csv
ratings.csv
README.txt
tags.csv


In [6]:
# Check the size of each dataset file

for item in MOVIELENS_DIR.iterdir():
    if item.is_file():
        size_mb = item.stat().st_size / (1024 ** 2)
        print(f"{item.name:15} {size_mb:,.2f} MB")

checksums.txt   0.00 MB
links.csv       1.86 MB
movies.csv      4.05 MB
ratings.csv     836.45 MB
README.txt      0.01 MB
tags.csv        69.00 MB


# Load the movie metadata

In [7]:
# Load the movie metadata

movies = pd.read_csv(MOVIELENS_DIR / "movies.csv")

print("Movies dataset loaded successfully.")
print("Shape:", movies.shape)

Movies dataset loaded successfully.
Shape: (87585, 3)


In [8]:
# Display the first five rows

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
# Check the column names

movies.columns

Index(['movieId', 'title', 'genres'], dtype='str')

In [10]:
# Check the data types and basic structure

movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int64
 1   title    87585 non-null  str  
 2   genres   87585 non-null  str  
dtypes: int64(1), str(2)
memory usage: 5.2 MB


In [11]:
# Check for missing values

movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [12]:
# Check for duplicate movie IDs

duplicate_movie_ids = movies["movieId"].duplicated().sum()

print("Duplicate movie IDs:", duplicate_movie_ids)

Duplicate movie IDs: 0


In [13]:
# Check for duplicate movie titles

duplicate_titles = movies["title"].duplicated().sum()

print("Duplicate movie titles:", duplicate_titles)

Duplicate movie titles: 203


In [14]:
# Examine movies with duplicate titles

duplicate_title_movies = movies[movies["title"].duplicated(keep=False)].sort_values("title")

duplicate_title_movies.head(20)

,movieId,title,genres
71461,230389,2020 (2020),Documentary
74770,245234,2020 (2020),Comedy
18413,96456,ATM (2012),Horror|Thriller
67307,217057,ATM (2012),Comedy|Romance
30796,136696,Absolution (2015),Action|Adventure|Crime|Thriller
37385,151797,Absolution (2015),Drama|Thriller
64233,209761,Adam (2019),Comedy
69506,223916,Adam (2019),Drama
580,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
22298,114240,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy


In [15]:
# Examine the different genre formats in the dataset

movies["genres"].value_counts().head(20)

genres
Drama                   12443
Documentary              8132
Comedy                   7761
(no genres listed)       7080
Comedy|Drama             3245
Drama|Romance            2825
Horror                   2487
Comedy|Romance           2229
Thriller                 1410
Comedy|Drama|Romance     1335
Drama|Thriller           1281
Horror|Thriller          1242
Crime|Drama              1150
Animation                1120
Drama|War                 818
Action                    789
Action|Drama              696
Western                   686
Crime|Drama|Thriller      655
Romance                   645
Name: count, dtype: int64

In [16]:
# Extract individual genres from the pipe-separated genre field

genre_lists = movies["genres"].str.split("|")

unique_genres = sorted(
    set(genre for genres in genre_lists for genre in genres)
)

print("Number of unique genre categories:", len(unique_genres))
print("\nGenres:")
print(unique_genres)

Number of unique genre categories: 20

Genres:
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [17]:
# Count the number of genres assigned to each movie

movies["genre_count"] = movies["genres"].str.split("|").str.len()

movies["genre_count"].value_counts().sort_index()

genre_count
1     44585
2     25262
3     13172
4      3521
5       849
6       161
7        32
8         2
10        1
Name: count, dtype: int64

In [18]:
# Remove the temporary analysis column

movies = movies.drop(columns=["genre_count"])

print("Current columns:", movies.columns.tolist())

Current columns: ['movieId', 'title', 'genres']


# Load the ratings dataset using memory-efficient data types

In [19]:
# Load the ratings dataset using memory-efficient data types

ratings = pd.read_csv(
    MOVIELENS_DIR / "ratings.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int64"
    }
)

print("Ratings dataset loaded successfully.")
print("Shape:", ratings.shape)

Ratings dataset loaded successfully.
Shape: (32000204, 4)


In [20]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [21]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int32  
 1   movieId    int32  
 2   rating     float32
 3   timestamp  int64  
dtypes: float32(1), int32(2), int64(1)
memory usage: 610.4 MB


In [22]:
# missing value check
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [23]:
# check duplicate rating record
duplicate_ratings = ratings.duplicated().sum()

print("Duplicate rating records:", duplicate_ratings)

Duplicate rating records: 0


In [24]:
# check rating values
ratings["rating"].value_counts().sort_index()

rating
0.5     525132
1.0     946675
1.5     531063
2.0    2028622
2.5    1685386
3.0    6054990
3.5    4290105
4.0    8367654
4.5    2974000
5.0    4596577
Name: count, dtype: int64

In [ ]:
# Find the number of unique users and movies
print("Unique users:", ratings["userId"].nunique())
print("Unique movies:", ratings["movieId"].nunique())

Unique users: 200948
Unique movies: 84432


In [26]:
# Check whether every rated movie has metadata
rated_movie_ids = set(ratings["movieId"].unique())
movie_metadata_ids = set(movies["movieId"].unique())

missing_metadata = rated_movie_ids - movie_metadata_ids

print("Rated movies without metadata:", len(missing_metadata))

Rated movies without metadata: 0


In [27]:
# Check whether every metadata movie has ratings
metadata_movie_ids = set(movies["movieId"].unique())
rated_movie_ids = set(ratings["movieId"].unique())

unrated_movies = metadata_movie_ids - rated_movie_ids

print("Movies with no ratings:", len(unrated_movies))

Movies with no ratings: 3153


In [28]:
# Check the timestamp range of the ratings dataset
print("Earliest timestamp:", ratings["timestamp"].min())
print("Latest timestamp:", ratings["timestamp"].max())

Earliest timestamp: 789652004
Latest timestamp: 1697164147


In [29]:
# Convert timestamps to dates
ratings["datetime"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s"
)

print("Earliest date:", ratings["datetime"].min())
print("Latest date:", ratings["datetime"].max())

Earliest date: 1995-01-09 11:46:44
Latest date: 2023-10-13 02:29:07


In [31]:
# Check how many ratings were given each year
ratings["year"] = ratings["datetime"].dt.year

ratings["year"].value_counts().sort_index()


year
1995          4
1996    1571368
1997     685388
1998     301691
1999    1174629
2000    1912322
2001    1160098
2002     849762
2003    1011000
2004    1139068
2005    1752573
2006    1141704
2007    1023813
2008    1117071
2009     890696
2010     860073
2011     729301
2012     695493
2013     563301
2014     518537
2015    1743866
2016    1918739
2017    1827953
2018    1391057
2019    1385467
2020    1688159
2021    1239912
2022     913105
2023     794054
Name: count, dtype: int64

In [37]:
# Check the number of rating per user
ratings_per_user = ratings.groupby("userId").size()

print("Average ratings per user:", ratings_per_user.mean())
print("Median ratings per user:", ratings_per_user.median())
print("Minimum ratings by a user:", ratings_per_user.min())

# Most active users
print("Maximum ratings by a user:", ratings_per_user.max())

Average ratings per user: 159.24619304496684
Median ratings per user: 73.0
Minimum ratings by a user: 20
Maximum ratings by a user: 33332


In [38]:
# Check movies popularity

ratings_per_movie = ratings.groupby("movieId").size()

print("Average ratings per movie:", ratings_per_movie.mean())
print("Median ratings per movie:", ratings_per_movie.median())
print("Minimum ratings for a movie:", ratings_per_movie.min())
print("Maximum ratings for a movie:", ratings_per_movie.max())

Average ratings per movie: 379.0056376729202
Median ratings per movie: 5.0
Minimum ratings for a movie: 1
Maximum ratings for a movie: 102929


In [39]:
#  Find the most-rated movies
top_rated_movies = (
    ratings_per_movie
    .sort_values(ascending=False)
    .head(10)
    .rename("rating_count")
    .reset_index()
    .merge(movies, on="movieId", how="left")
)

top_rated_movies[["movieId", "title", "rating_count"]]

,movieId,title,rating_count
0,318,"Shawshank Redemption, The (1994)",102929
1,356,Forrest Gump (1994),100296
2,296,Pulp Fiction (1994),98409
3,2571,"Matrix, The (1999)",93808
4,593,"Silence of the Lambs, The (1991)",90330
5,260,Star Wars: Episode IV - A New Hope (1977),85010
6,2959,Fight Club (1999),77332
7,480,Jurassic Park (1993),75233
8,527,Schindler's List (1993),73849
9,4993,"Lord of the Rings: The Fellowship of the Ring,...",73122


In [40]:
# Calculate average rating per movie
average_rating_per_movie = ratings.groupby("movieId")["rating"].mean()

print("Overall average rating:", ratings["rating"].mean())
print("Median movie average rating:", average_rating_per_movie.median())

Overall average rating: 3.5403955
Median movie average rating: 3.0714285


In [41]:
# Find movies with very few ratings
movies_with_counts = (
    ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        average_rating=("rating", "mean")
    )
    .reset_index()
    .merge(movies, on="movieId", how="left")
)

movies_with_counts.sort_values(
    "average_rating",
    ascending=False
).head(10)[
    ["movieId", "title", "rating_count", "average_rating"]
]

,movieId,title,rating_count,average_rating
81273,282081,Shadows (2023),1,5.0
81271,282073,Mal de Ojo (2022),1,5.0
81251,282021,Mr. Krueger's Christmas (1980),1,5.0
81250,282019,It's a Wonderful Life (2007),1,5.0
54558,193529,Traces of Smoke (1992),2,5.0
54572,193557,Love Pret-a-porte (2017),1,5.0
54814,194066,The Revolution That Wasn't (2008),1,5.0
54654,193727,Still Burning (2016),1,5.0
81226,281956,ManFish (2022),1,5.0
81199,281902,Going Varsity In Mariachi,1,5.0


In [42]:
# Examine the opposite: movies with lots of ratings
popular_movies = movies_with_counts[
    movies_with_counts["rating_count"] >= 1000
]

popular_movies.sort_values(
    "average_rating",
    ascending=False
).head(10)[
    ["movieId", "title", "rating_count", "average_rating"]
]

,movieId,title,rating_count,average_rating
44190,171011,Planet Earth II (2016),1956,4.446830
39306,159817,Planet Earth (2006),2948,4.444369
44042,170705,Band of Brothers (2001),2811,4.426538
314,318,"Shawshank Redemption, The (1994)",102929,4.404614
840,858,"Godfather, The (1972)",66440,4.317030
58698,202439,Parasite (2019),11670,4.312253
47975,179135,Blue Planet II (2017),1163,4.300086
56760,198185,Twin Peaks (1989),1140,4.298684
1173,1203,12 Angry Men (1957),21863,4.265311
49,50,"Usual Suspects, The (1995)",67750,4.265070


In [44]:
# Check user rating behavior
average_rating_per_user = ratings.groupby("userId")["rating"].mean()

print("Overall average user rating:", average_rating_per_user.mean())
print("Median user average rating:", average_rating_per_user.median())
print("Minimum user average rating:", average_rating_per_user.min())
print("Maximum user average rating:", average_rating_per_user.max())

Overall average user rating: 3.7043686
Median user average rating: 3.725
Minimum user average rating: 0.5
Maximum user average rating: 5.0


In [45]:
# How user rating habits are distributed

user_rating_summary = pd.cut(
    average_rating_per_user,
    bins=[0, 2, 3, 3.5, 4, 4.5, 5],
    include_lowest=True
).value_counts().sort_index()

user_rating_summary

rating
(-0.001, 2.0]      833
(2.0, 3.0]       13578
(3.0, 3.5]       48477
(3.5, 4.0]       85049
(4.0, 4.5]       45057
(4.5, 5.0]        7954
Name: count, dtype: int64

In [46]:
# Extract movie release year
movies["release_year"] = (
    movies["title"]
    .str.extract(r"\((\d{4})\)")
    .astype("Int64")
)

print("Earliest release year:", movies["release_year"].min())
print("Latest release year:", movies["release_year"].max())
print("Movies without a release year:", movies["release_year"].isna().sum())

Earliest release year: 1874
Latest release year: 2023
Movies without a release year: 615


In [47]:
# Inspect the oldest movies
movies.sort_values("release_year").head(20)[
    ["movieId", "title", "release_year", "genres"]
]

,movieId,title,release_year,genres
35829,148054,Passage de Venus (1874),1874,Documentary
82012,275697,The Magic Rosette (1878),1878,Animation
35826,148048,Sallie Gardner at a Gallop (1878),1878,(no genres listed)
60912,202045,Athlete Swinging a Pick (1880),1880,Documentary
82013,275699,The Kiss (1882),1882,Documentary
44258,166800,Buffalo Running (1883),1883,(no genres listed)
82017,275709,L'Homme Machine (1885),1885,Animation
82016,275705,Semi-Nude Woman Hopping on Left Foot (1887),1887,Documentary
35822,148040,Man Walking Around a Corner (1887),1887,(no genres listed)
82015,275703,Boys Playing Leapfrog: Side View (1887),1887,Documentary


In [48]:
# Inspect movies with missing release years
movies[movies["release_year"].isna()][
    ["movieId", "title", "genres"]
].head(20)

,movieId,title,genres
15038,79607,"Millions Game, The (Das Millionenspiel)",Action|Drama|Sci-Fi|Thriller
25425,123619,Terrible Joe Moran,(no genres listed)
26356,125571,The Court-Martial of Jackie Robinson,(no genres listed)
26381,125632,In Our Garden,(no genres listed)
26466,125958,Stephen Fry In America - New World,(no genres listed)
26650,126438,Two: The Story of Roman & Nyro,Documentary|Drama
26773,127005,A Year Along the Abandoned Road,(no genres listed)
27301,128612,Body/Cialo,Comedy|Drama|Mystery
27354,128734,Polskie gówno,Comedy|Musical
27666,129651,The Third Reich: The Rise & Fall,(no genres listed)


In [49]:
# Count individual genres
genre_counts = (
    movies["genres"]
    .str.split("|")
    .explode()
    .value_counts()
)

genre_counts

genres
Drama                 34175
Comedy                23124
Thriller              11823
Romance               10369
Action                 9668
Documentary            9363
Horror                 8654
(no genres listed)     7080
Crime                  6976
Adventure              5402
Sci-Fi                 4907
Animation              4617
Children               4520
Mystery                4013
Fantasy                3851
War                    2325
Western                1696
Musical                1059
Film-Noir               353
IMAX                    195
Name: count, dtype: int64

# Inspect the tags dataset

In [50]:
#Load tags dataset
tags = pd.read_csv(
    MOVIELENS_DIR / "tags.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "tag": "string",
        "timestamp": "int64"
    }
)

print("Tags dataset loaded successfully.")
print("Shape:", tags.shape)

Tags dataset loaded successfully.
Shape: (2000072, 4)


In [51]:
tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [52]:
# Check tag data quality
print("Missing values:")
print(tags.isnull().sum())

print("\nDuplicate tag records:", tags.duplicated().sum())

Missing values:
userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

Duplicate tag records: 0


In [53]:
# How many unique tags
print("Unique tags:", tags["tag"].nunique())
print("Unique users who created tags:", tags["userId"].nunique())
print("Unique movies with tags:", tags["movieId"].nunique())

Unique tags: 140979
Unique users who created tags: 15848
Unique movies with tags: 51323


In [54]:
# Inspect the most commonm tags
tag_counts = (
    tags["tag"]
    .dropna()
    .value_counts()
)

tag_counts.head(30)

tag
sci-fi                10996
atmospheric            9589
action                 8473
comedy                 8139
funny                  7467
surreal                7231
visually appealing     7090
based on a book        6617
twist ending           6521
dark comedy            6053
thought-provoking      6000
dystopia               5646
violence               5389
cinematography         5277
romance                5271
murder                 5180
social commentary      5152
stylized               4917
fantasy                4844
psychology             4827
time travel            4682
quirky                 4650
dark                   4529
classic                4482
adventure              4387
coming of age          4350
friendship             4257
suspense               4227
revenge                4200
BD-R                   3967
Name: count, dtype: int64[pyarrow]

In [55]:
# Check whether tags connect to valid movies
tagged_movie_ids = set(tags["movieId"].unique())
movie_ids = set(movies["movieId"].unique())

missing_tag_movie_metadata = tagged_movie_ids - movie_ids

print(
    "Tagged movies without movie metadata:",
    len(missing_tag_movie_metadata)
)

Tagged movies without movie metadata: 0


# Inspect the links dataset

In [56]:
# load the links dataset
links = pd.read_csv(
    MOVIELENS_DIR / "links.csv",
    dtype={
        "movieId": "int32",
        "imdbId": "Int64",
        "tmdbId": "Int64"
    }
)

print("Links dataset loaded successfully.")
print("Shape:", links.shape)

Links dataset loaded successfully.
Shape: (87585, 3)


In [57]:
# Inspect the links
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [58]:
# Check missing external IDs
print("Missing values:")
print(links.isnull().sum())

Missing values:
movieId      0
imdbId       0
tmdbId     124
dtype: int64


In [59]:
# Check duplicate IDs in the link tables
print("Duplicate MovieLens IDs:", links["movieId"].duplicated().sum())
print("Duplicate IMDb IDs:", links["imdbId"].duplicated().sum())
print("Duplicate TMDB IDs:", links["tmdbId"].duplicated().sum())

Duplicate MovieLens IDs: 0
Duplicate IMDb IDs: 0
Duplicate TMDB IDs: 159


In [60]:
# Inspect duplicate TMDB mappings
duplicate_tmdb_ids = (
    links.loc[links["tmdbId"].duplicated(keep=False)]
    .sort_values("tmdbId")
)

duplicate_tmdb_ids.head(20)

,movieId,imdbId,tmdbId
5892,6003,290538,4912
34340,144606,270288,4912
10103,34330,368089,9775
49943,178755,376800,9775
47959,174533,235679,10991
4138,4241,266860,10991
5562,5672,313487,12600
47964,174543,287635,12600
61157,202599,1016268,13020
9897,33154,413845,13020


In [61]:
# Final relationship check
print("Movies:", movies["movieId"].nunique())
print("Rated movies:", ratings["movieId"].nunique())
print("Tagged movies:", tags["movieId"].nunique())

print("\nRated movies missing from movies:",
      len(set(ratings["movieId"]) - set(movies["movieId"])))

print("Tagged movies missing from movies:",
      len(set(tags["movieId"]) - set(movies["movieId"])))

print("Ratings users:", ratings["userId"].nunique())
print("Tagging users:", tags["userId"].nunique())

Movies: 87585
Rated movies: 84432
Tagged movies: 51323

Rated movies missing from movies: 0
Tagged movies missing from movies: 0
Ratings users: 200948
Tagging users: 15848


In [62]:
# Save a simple data-understanding summary
data_summary = {
    "movies": len(movies),
    "rated_movies": ratings["movieId"].nunique(),
    "unrated_movies": len(movies) - ratings["movieId"].nunique(),
    "users": ratings["userId"].nunique(),
    "ratings": len(ratings),
    "tags": len(tags),
    "unique_tags": tags["tag"].nunique(),
    "tagged_movies": tags["movieId"].nunique(),
    "missing_movie_genres": (movies["genres"] == "(no genres listed)").sum(),
    "missing_release_year": movies["release_year"].isna().sum(),
    "missing_tmdb_ids": links["tmdbId"].isna().sum(),
    "duplicate_movie_ids": movies["movieId"].duplicated().sum(),
    "duplicate_rating_records": ratings.duplicated().sum(),
}

data_summary

{'movies': 87585,
 'rated_movies': 84432,
 'unrated_movies': 3153,
 'users': 200948,
 'ratings': 32000204,
 'tags': 2000072,
 'unique_tags': 140979,
 'tagged_movies': 51323,
 'missing_movie_genres': np.int64(7080),
 'missing_release_year': np.int64(615),
 'missing_tmdb_ids': np.int64(124),
 'duplicate_movie_ids': np.int64(0),
 'duplicate_rating_records': np.int64(0)}

## Key Findings and Data Quality Decisions

### Dataset Overview

The MovieLens 32M dataset contains:

* 87,585 movies
* 200,948 users
* 32,000,204 ratings
* 2,000,072 user-generated tags
* 140,979 unique tags

The dataset is sufficiently large and diverse for developing and evaluating a recommendation system.

### Data Quality Findings

* No missing values were found in the `movies` dataset.
* No duplicate `movieId` values were found.
* Duplicate movie titles exist, but they represent different movies in some cases. Therefore, `movieId` will be used as the primary movie identifier.
* The ratings dataset contains no missing values and no duplicate rating records.
* Ratings range from 0.5 to 5.0 in increments of 0.5.
* Every rated movie has a corresponding record in the movie metadata.
* 3,153 movies have no ratings. These movies may still be useful for content-based recommendations.
* 7,080 movies have `(no genres listed)`.
* 615 movies do not have an identifiable release year from the movie title.
* Only 124 movies have missing TMDB IDs.
* Some TMDB IDs appear more than once, so MovieLens `movieId` will remain the primary identifier rather than assuming TMDB IDs are unique.

### Rating Behavior

The rating data is highly uneven:

* The average user has approximately 159 ratings, while the median user has 73.
* Some users have rated thousands of movies.
* The average movie has approximately 379 ratings, but the median movie has only 5 ratings.
* Therefore, movie popularity is highly skewed.
* The overall average rating is approximately 3.54.
* Movies with very few ratings can have extreme average ratings, so average rating alone should not be used to identify the highest-quality movies.

### Metadata Observations

The dataset contains 20 genre categories, including the `(no genres listed)` placeholder.

Drama and Comedy are the most common genres, while some genres such as Film-Noir and IMAX occur much less frequently.

User-generated tags provide additional information about movies, including themes, moods, styles, narrative characteristics, and other descriptive concepts. These tags may be useful as features for the content-based recommendation model.

### Data Quality Decisions

At this stage, no major records will be deleted simply because they have unusual values.

The following decisions will be made during preprocessing:

1. Use `movieId` as the primary movie identifier.
2. Retain movies without ratings because they may be useful for content-based recommendations.
3. Handle `(no genres listed)` appropriately during content-based preprocessing rather than automatically deleting those movies.
4. Investigate movies with missing release years before deciding whether external metadata is necessary.
5. Retain the original rating scale of 0.5–5.0.
6. Consider user and movie activity levels when designing filtering rules for collaborative filtering.
7. Treat popularity and average rating as separate concepts.
8. Consider the strong sparsity and popularity imbalance when selecting recommendation algorithms and evaluation methods.

These findings will guide the exploratory analysis, preprocessing, recommendation models, and evaluation stages that follow.
